
# Task 1 — MICN for Traffic Forecasting After Incidents

This notebook implements a compact **MICN-style multiscale convolutional forecasting model** for the supplied TraffiDent-style dataset.

## Data structure

Each row contains:

- `384` input values = `24 timesteps × 16 features`
- `6` targets = traffic flow at `t+1, ..., t+6`

The experiments predict:

- `t+1`
- `t+3`
- `t+6`

## Three experimental cases

1. **MICN-General**  
   Uses non-incident traffic, weather, temporal, and road features.

2. **MICN-Binary**  
   Uses the same General input plus one binary incident indicator.

3. **MICN-TypeEmb**  
   Uses the same General input plus a learnable 16-dimensional incident-type embedding.  
   Non-incident samples receive a zero vector.

## TraffiDent-style evaluation

Each trained model is evaluated on:

- all test samples;
- incident-only test samples.

Metrics:

- MAE
- RMSE
- MAPE

> This is an adaptation of the TraffiDent forecasting workflow to hourly, pre-windowed data. MICN was not one of the original forecasting baselines in the TraffiDent table.


In [14]:

# 1. Imports

from pathlib import Path
import copy
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

print("PyTorch version:", torch.__version__)


PyTorch version: 2.12.1


## 2. Reproducibility and configuration

In [15]:

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

TRAIN_PATH = "/Users/andishahifahmuthahharah/Downloads/traffic_data_24_6/train_24_6.csv"
VAL_PATH   = "/Users/andishahifahmuthahharah/Downloads/traffic_data_24_6/val_24_6.csv"
TEST_PATH  = "/Users/andishahifahmuthahharah/Downloads/traffic_data_24_6/test_24_6.csv"

N_TIMESTEPS = 24
N_FEATURES = 16
N_INPUT_COLUMNS = 384
N_TARGET_COLUMNS = 6
EXPECTED_COLUMNS = 390

HORIZONS = [1, 3, 6]
TARGET_INDICES = [0, 2, 5]

TOTAL_FLOW_IDX = 0
IMPACT_SEQUENCE_HOUR_IDX = 5
INCIDENT_TYPE_IDX = 12

# General features, using the column description supplied by the user:
# offsets 1-5, 9-10, 11, 14-15
# zero-based indices: 0-4, 8-10, 13-14
GENERAL_FEATURE_INDICES = [0, 1, 2, 3, 4, 8, 9, 10, 13, 14]

INCIDENT_EMBED_DIM = 16
NUM_INCIDENT_CODES = 9  # 0 through 8

BATCH_SIZE = 256
MAX_EPOCHS = 50
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
D_MODEL = 64
DROPOUT = 0.1
NUM_WORKERS = 0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

OUTPUT_DIR = Path("micn_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("General input features:", len(GENERAL_FEATURE_INDICES))


Device: mps
General input features: 10


## 3. Feature metadata

In [16]:

FEATURE_NAMES = [
    "total_flow",
    "precipitation",
    "temperature_2m",
    "wind_gusts_10m",
    "relative_humidity",
    "impact_sequence_hour",
    "is_major_incident",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "feat_12",
    "incident_type",
    "lane_count",
    "road_functional_hierarchy",
    "distance_to_intersection",
]

INCIDENT_TYPE_MAP = {
    0: "NO_INCIDENT",
    1: "CRASH",
    2: "BREAKDOWN",
    3: "HAZARD",
    4: "ROADWORK",
    5: "TRAFFIC_CONTROL",
    6: "ADVERSE_WEATHER",
    7: "EVENT",
    8: "OTHERS",
}

print("General features:")
for idx in GENERAL_FEATURE_INDICES:
    print(f"- {idx + 1}: {FEATURE_NAMES[idx]}")


General features:
- 1: total_flow
- 2: precipitation
- 3: temperature_2m
- 4: wind_gusts_10m
- 5: relative_humidity
- 9: hour_sin
- 10: hour_cos
- 11: is_weekend
- 14: lane_count
- 15: road_functional_hierarchy


## 4. Load headerless matrices

In [17]:

def load_matrix(path, split_name):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{split_name}: file not found: {path.resolve()}"
        )

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, header=None)
    elif path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path, header=None)
    elif path.suffix.lower() in [".parquet", ".pq"]:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    if df.shape[1] != EXPECTED_COLUMNS:
        raise ValueError(
            f"{split_name}: expected {EXPECTED_COLUMNS} columns, "
            f"found {df.shape[1]}"
        )

    df = df.apply(pd.to_numeric, errors="coerce")
    if df.isna().any().any():
        raise ValueError(
            f"{split_name}: missing or non-numeric values found."
        )

    return df


train_raw = load_matrix(TRAIN_PATH, "train")
val_raw = load_matrix(VAL_PATH, "validation")
test_raw = load_matrix(TEST_PATH, "test")

print("Train:", train_raw.shape)
print("Validation:", val_raw.shape)
print("Test:", test_raw.shape)


Train: (603865, 390)
Validation: (206425, 390)
Test: (187105, 390)


## 5. Extract tensors and incident labels

In [18]:

def extract_split(df, split_name):
    values = df.to_numpy(dtype=np.float32)

    X_all = values[:, :N_INPUT_COLUMNS].reshape(
        -1, N_TIMESTEPS, N_FEATURES
    )
    y_six = values[:, N_INPUT_COLUMNS:]
    y = y_six[:, TARGET_INDICES]

    # Input used in all three cases
    X_general = X_all[:, :, GENERAL_FEATURE_INDICES]

    # Incident status at anchor timestep t
    impact_at_t = X_all[:, -1, IMPACT_SEQUENCE_HOUR_IDX]
    incident_binary = (impact_at_t >= 0).astype(np.int64)

    # Incident category at anchor timestep t
    incident_type = np.rint(
        X_all[:, -1, INCIDENT_TYPE_IDX]
    ).astype(np.int64)

    # Force non-incident samples to category 0
    incident_type = np.where(
        incident_binary == 0,
        0,
        incident_type
    )

    invalid = ~np.isin(incident_type, np.arange(9))
    if invalid.any():
        raise ValueError(
            f"{split_name}: invalid incident codes "
            f"{np.unique(incident_type[invalid])}"
        )

    return {
        "X_general": X_general.astype(np.float32),
        "y": y.astype(np.float32),
        "incident_binary": incident_binary,
        "incident_type": incident_type,
    }


train_data = extract_split(train_raw, "train")
val_data = extract_split(val_raw, "validation")
test_data = extract_split(test_raw, "test")

print("X_general:", train_data["X_general"].shape)
print("y:", train_data["y"].shape)
print("Incident rate:", train_data["incident_binary"].mean())


X_general: (603865, 24, 10)
y: (603865, 3)
Incident rate: 0.002974174691363136


## 6. Dataset class and data loaders

In [19]:

class TrafficDataset(Dataset):
    def __init__(self, data):
        self.X = torch.tensor(data["X_general"], dtype=torch.float32)
        self.y = torch.tensor(data["y"], dtype=torch.float32)
        self.binary = torch.tensor(
            data["incident_binary"], dtype=torch.long
        )
        self.incident_type = torch.tensor(
            data["incident_type"], dtype=torch.long
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "x": self.X[idx],
            "y": self.y[idx],
            "binary": self.binary[idx],
            "incident_type": self.incident_type[idx],
        }


def make_loader(data, shuffle=False):
    return DataLoader(
        TrafficDataset(data),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )


train_loader = make_loader(train_data, shuffle=True)
val_loader = make_loader(val_data, shuffle=False)
test_loader = make_loader(test_data, shuffle=False)


## 7. MICN-style architecture

In [20]:

class MovingAverage(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.pool = nn.AvgPool1d(
            kernel_size=kernel_size,
            stride=1,
            padding=0
        )

    def forward(self, x):
        # x: [B, T, C]
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, pad, 1)
        padded = torch.cat([front, x, end], dim=1)
        trend = self.pool(padded.transpose(1, 2)).transpose(1, 2)
        return trend


class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size=5):
        super().__init__()
        self.moving_average = MovingAverage(kernel_size)

    def forward(self, x):
        trend = self.moving_average(x)
        seasonal = x - trend
        return seasonal, trend


class MultiScaleIsometricBlock(nn.Module):
    def __init__(self, d_model, kernels=(3, 5, 7), dropout=0.1):
        super().__init__()
        self.branches = nn.ModuleList()

        for kernel in kernels:
            padding = kernel // 2
            self.branches.append(
                nn.Sequential(
                    nn.Conv1d(
                        d_model,
                        d_model,
                        kernel_size=kernel,
                        padding=padding,
                        groups=d_model
                    ),
                    nn.GELU(),
                    nn.Conv1d(d_model, d_model, kernel_size=1),
                    nn.GELU(),
                    nn.Dropout(dropout),
                )
            )

        self.fusion = nn.Sequential(
            nn.Conv1d(
                d_model * len(kernels),
                d_model,
                kernel_size=1
            ),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: [B, T, D]
        residual = x
        z = x.transpose(1, 2)
        branch_outputs = [branch(z) for branch in self.branches]
        fused = self.fusion(torch.cat(branch_outputs, dim=1))
        fused = fused.transpose(1, 2)
        return self.norm(residual + fused)


class MICNForecaster(nn.Module):
    def __init__(
        self,
        input_dim,
        mode="general",
        d_model=64,
        embed_dim=16,
        dropout=0.1
    ):
        super().__init__()
        if mode not in {"general", "binary", "typeemb"}:
            raise ValueError("Invalid mode")

        self.mode = mode
        self.decomposition = SeriesDecomposition(kernel_size=5)

        self.seasonal_projection = nn.Linear(input_dim, d_model)
        self.trend_projection = nn.Linear(input_dim, d_model)

        self.mic_blocks = nn.Sequential(
            MultiScaleIsometricBlock(
                d_model, kernels=(3, 5, 7), dropout=dropout
            ),
            MultiScaleIsometricBlock(
                d_model, kernels=(3, 5, 7), dropout=dropout
            ),
        )

        self.temporal_pool = nn.AdaptiveAvgPool1d(1)

        aux_dim = 0
        if mode == "binary":
            aux_dim = 1
        elif mode == "typeemb":
            self.type_embedding = nn.Embedding(
                NUM_INCIDENT_CODES,
                embed_dim,
                padding_idx=0
            )
            aux_dim = embed_dim

        self.head = nn.Sequential(
            nn.Linear(d_model * 2 + aux_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, len(HORIZONS)),
        )

    def forward(self, x, binary, incident_type):
        seasonal, trend = self.decomposition(x)

        seasonal = self.seasonal_projection(seasonal)
        trend = self.trend_projection(trend)

        seasonal = self.mic_blocks(seasonal)

        seasonal_repr = self.temporal_pool(
            seasonal.transpose(1, 2)
        ).squeeze(-1)

        trend_repr = self.temporal_pool(
            trend.transpose(1, 2)
        ).squeeze(-1)

        representation = torch.cat(
            [seasonal_repr, trend_repr], dim=1
        )

        if self.mode == "binary":
            representation = torch.cat(
                [representation, binary.float().unsqueeze(1)],
                dim=1
            )

        elif self.mode == "typeemb":
            type_vec = self.type_embedding(incident_type)
            # Explicit zero masking for non-incident samples
            type_vec = type_vec * binary.float().unsqueeze(1)
            representation = torch.cat(
                [representation, type_vec],
                dim=1
            )

        return self.head(representation)


## 8. Training and early stopping

In [21]:

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_count = 0
    criterion = nn.L1Loss()

    for batch in loader:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        binary = batch["binary"].to(DEVICE)
        incident_type = batch["incident_type"].to(DEVICE)

        if training:
            optimizer.zero_grad()

        pred = model(x, binary, incident_type)
        loss = criterion(pred, y)

        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            optimizer.step()

        batch_size = len(y)
        total_loss += loss.item() * batch_size
        total_count += batch_size

    return total_loss / total_count


def train_model(mode):
    seed_everything(SEED)

    model = MICNForecaster(
        input_dim=len(GENERAL_FEATURE_INDICES),
        mode=mode,
        d_model=D_MODEL,
        embed_dim=INCIDENT_EMBED_DIM,
        dropout=DROPOUT,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_state = None
    best_val = float("inf")
    patience_count = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = run_epoch(
            model, train_loader, optimizer
        )
        with torch.no_grad():
            val_loss = run_epoch(model, val_loader)

        history.append({
            "epoch": epoch,
            "train_mae_loss": train_loss,
            "val_mae_loss": val_loss,
        })

        print(
            f"{mode:8s} | epoch {epoch:02d} | "
            f"train {train_loss:.5f} | val {val_loss:.5f}"
        )

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            print("Early stopping")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


## 9. Prediction and metrics

In [22]:

@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    predictions = []
    targets = []
    binaries = []
    types = []

    for batch in loader:
        x = batch["x"].to(DEVICE)
        binary = batch["binary"].to(DEVICE)
        incident_type = batch["incident_type"].to(DEVICE)

        pred = model(x, binary, incident_type)

        predictions.append(pred.cpu().numpy())
        targets.append(batch["y"].numpy())
        binaries.append(batch["binary"].numpy())
        types.append(batch["incident_type"].numpy())

    return {
        "pred": np.concatenate(predictions),
        "y": np.concatenate(targets),
        "binary": np.concatenate(binaries),
        "incident_type": np.concatenate(types),
    }


def safe_mape(y_true, y_pred, epsilon=1e-6):
    mask = np.abs(y_true) > epsilon
    if mask.sum() == 0:
        return np.nan
    return np.mean(
        np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
    ) * 100


def evaluate_prediction(result, model_name, subset_name, mask=None):
    if mask is None:
        mask = np.ones(len(result["y"]), dtype=bool)

    rows = []
    for j, horizon in enumerate(HORIZONS):
        y_true = result["y"][mask, j]
        y_pred = result["pred"][mask, j]

        rows.append({
            "model": model_name,
            "subset": subset_name,
            "horizon_hour": horizon,
            "n_samples": len(y_true),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "MAPE_percent": safe_mape(y_true, y_pred),
        })

    return pd.DataFrame(rows)


## 10. Run the three MICN experiments

In [23]:

EXPERIMENTS = {
    "MICN-General": "general",
    "MICN-Binary": "binary",
    "MICN-TypeEmb": "typeemb",
}

all_metrics = []
all_histories = []
all_predictions = []
trained_models = {}

for model_name, mode in EXPERIMENTS.items():
    print("\n" + "=" * 70)
    print("Training:", model_name)

    start = time.time()
    model, history = train_model(mode)
    elapsed = time.time() - start

    trained_models[model_name] = model
    history["model"] = model_name
    history["training_seconds"] = elapsed
    all_histories.append(history)

    # Validation
    val_result = predict_model(model, val_loader)
    all_metrics.append(
        evaluate_prediction(
            val_result, model_name, "validation_all"
        )
    )
    all_metrics.append(
        evaluate_prediction(
            val_result,
            model_name,
            "validation_incident_only",
            mask=val_result["binary"] == 1
        )
    )

    # Test
    test_result = predict_model(model, test_loader)
    all_metrics.append(
        evaluate_prediction(
            test_result, model_name, "test_all"
        )
    )
    all_metrics.append(
        evaluate_prediction(
            test_result,
            model_name,
            "test_incident_only",
            mask=test_result["binary"] == 1
        )
    )

    pred_df = pd.DataFrame({
        "sample_index": np.arange(len(test_result["y"])),
        "model": model_name,
        "incident_binary": test_result["binary"],
        "incident_type_code": test_result["incident_type"],
        "incident_type": [
            INCIDENT_TYPE_MAP[int(v)]
            for v in test_result["incident_type"]
        ],
    })

    for j, horizon in enumerate(HORIZONS):
        pred_df[f"actual_t+{horizon}"] = test_result["y"][:, j]
        pred_df[f"pred_t+{horizon}"] = test_result["pred"][:, j]

    all_predictions.append(pred_df)

metrics_df = pd.concat(all_metrics, ignore_index=True)
history_df = pd.concat(all_histories, ignore_index=True)
predictions_df = pd.concat(all_predictions, ignore_index=True)

display(
    metrics_df.sort_values(
        ["subset", "horizon_hour", "MAE"]
    ).reset_index(drop=True)
)



Training: MICN-General
general  | epoch 01 | train 0.32393 | val 0.29401
general  | epoch 02 | train 0.23082 | val 0.26863
general  | epoch 03 | train 0.21512 | val 0.24953
general  | epoch 04 | train 0.20577 | val 0.24771
general  | epoch 05 | train 0.19985 | val 0.23225
general  | epoch 06 | train 0.19470 | val 0.24252
general  | epoch 07 | train 0.19148 | val 0.22746
general  | epoch 08 | train 0.18858 | val 0.22285
general  | epoch 09 | train 0.18621 | val 0.22039
general  | epoch 10 | train 0.18404 | val 0.21651
general  | epoch 11 | train 0.18237 | val 0.21812
general  | epoch 12 | train 0.18100 | val 0.21059
general  | epoch 13 | train 0.17973 | val 0.21153
general  | epoch 14 | train 0.17870 | val 0.21814
general  | epoch 15 | train 0.17763 | val 0.20729
general  | epoch 16 | train 0.17640 | val 0.20416
general  | epoch 17 | train 0.17587 | val 0.20450
general  | epoch 18 | train 0.17509 | val 0.20951
general  | epoch 19 | train 0.17454 | val 0.20805
general  | epoch 20 | trai

,model,subset,horizon_hour,n_samples,MAE,RMSE,MAPE_percent
0,MICN-Binary,test_all,1,187105,0.129284,0.292227,196.888208
1,MICN-General,test_all,1,187105,0.132233,0.293638,221.314168
2,MICN-TypeEmb,test_all,1,187105,0.132683,0.301771,234.075618
3,MICN-Binary,test_all,3,187105,0.183333,0.424981,329.368591
4,MICN-General,test_all,3,187105,0.184075,0.424391,286.119223
5,MICN-TypeEmb,test_all,3,187105,0.186195,0.432590,257.320094
6,MICN-General,test_all,6,187105,0.241565,0.566781,660.129595
7,MICN-TypeEmb,test_all,6,187105,0.243841,0.569416,569.439888
8,MICN-Binary,test_all,6,187105,0.244563,0.561354,808.946800
9,MICN-General,test_incident_only,1,617,0.111026,0.290026,39.590710


## 11. Marginal value of incident information

In [24]:

def build_delta_table(metrics, subset):
    selected = metrics[metrics["subset"] == subset]

    pivot = selected.pivot_table(
        index="horizon_hour",
        columns="model",
        values="MAE",
        aggfunc="first"
    )

    pivot["Delta1_General_minus_Binary"] = (
        pivot["MICN-General"] - pivot["MICN-Binary"]
    )
    pivot["Delta2_Binary_minus_TypeEmb"] = (
        pivot["MICN-Binary"] - pivot["MICN-TypeEmb"]
    )

    return pivot.reset_index()


delta_all = build_delta_table(metrics_df, "test_all")
delta_incident = build_delta_table(
    metrics_df, "test_incident_only"
)

print("All test samples")
display(delta_all)

print("Incident-only test samples")
display(delta_incident)


All test samples


model,horizon_hour,MICN-Binary,MICN-General,MICN-TypeEmb,Delta1_General_minus_Binary,Delta2_Binary_minus_TypeEmb
0,1,0.129284,0.132233,0.132683,0.002949,-0.003398
1,3,0.183333,0.184075,0.186195,0.000742,-0.002863
2,6,0.244563,0.241565,0.243841,-0.002998,0.000722


Incident-only test samples


model,horizon_hour,MICN-Binary,MICN-General,MICN-TypeEmb,Delta1_General_minus_Binary,Delta2_Binary_minus_TypeEmb
0,1,0.117725,0.111026,0.118653,-0.006699,-0.000928
1,3,0.172204,0.170210,0.175386,-0.001994,-0.003182
2,6,0.216571,0.223246,0.226766,0.006675,-0.010195


## 12. Final result table

In [25]:

final_table = (
    metrics_df[
        metrics_df["subset"].isin(
            ["test_all", "test_incident_only"]
        )
    ]
    .pivot_table(
        index=["model", "subset"],
        columns="horizon_hour",
        values=["MAE", "RMSE", "MAPE_percent"],
        aggfunc="first"
    )
    .round(4)
)

display(final_table)


MAE                 MAPE_percent  \
horizon_hour                          1       3       6            1   
model        subset                                                    
MICN-Binary  test_all            0.1293  0.1833  0.2446     196.8882   
             test_incident_only  0.1177  0.1722  0.2166      42.1834   
MICN-General test_all            0.1322  0.1841  0.2416     221.3142   
             test_incident_only  0.1110  0.1702  0.2232      39.5907   
MICN-TypeEmb test_all            0.1327  0.1862  0.2438     234.0756   
             test_incident_only  0.1187  0.1754  0.2268      48.0108   

                                                       RMSE                  
horizon_hour                            3         6       1       3       6  
model        subset                                                          
MICN-Binary  test_all            329.3686  808.9468  0.2922  0.4250  0.5614  
             test_incident_only   36.7846  138.1557  0.2906  0.4028  0.5345  
MICN-General test_all            286.1192  660.1296  0.2936  0.4244  0.5668  
             test_incident_only   33.5594  133.4426  0.2900  0.4166  0.5839  
MICN-TypeEmb test_all            257.3201  569.4399  0.3018  0.4326  0.5694  
             test_incident_only   36.0188  139.4681  0.3016  0.4197  0.5402

## 13. Save models and outputs

In [ ]:

metrics_df.to_csv(
    OUTPUT_DIR / "micn_metrics.csv", index=False
)
history_df.to_csv(
    OUTPUT_DIR / "micn_training_history.csv", index=False
)
predictions_df.to_csv(
    OUTPUT_DIR / "micn_test_predictions.csv", index=False
)
delta_all.to_csv(
    OUTPUT_DIR / "micn_delta_test_all.csv", index=False
)
delta_incident.to_csv(
    OUTPUT_DIR / "micn_delta_test_incident_only.csv",
    index=False
)

for model_name, model in trained_models.items():
    safe_name = model_name.lower().replace("-", "_")
    torch.save(
        model.state_dict(),
        OUTPUT_DIR / f"{safe_name}.pt"
    )

config = {
    "seed": SEED,
    "horizons": HORIZONS,
    "general_feature_indices_zero_based": GENERAL_FEATURE_INDICES,
    "general_feature_names": [
        FEATURE_NAMES[i] for i in GENERAL_FEATURE_INDICES
    ],
    "incident_embedding_dim": INCIDENT_EMBED_DIM,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "d_model": D_MODEL,
    "dropout": DROPOUT,
    "incident_type_mapping": INCIDENT_TYPE_MAP,
}

with open(
    OUTPUT_DIR / "micn_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)

print("Saved to:", OUTPUT_DIR.resolve())



## Interpretation guidance

Use:

- **Section 12** as the main result table.
- **Section 11** to determine whether incident presence or type improves prediction.
- Focus primarily on `test_incident_only` for post-incident forecasting.

Interpret deltas as:

- positive `Delta1`: Binary improves over General;
- positive `Delta2`: TypeEmb improves over Binary.

Because the targets are Z-score normalized, MAE and RMSE are in normalized units. MAPE may be unstable when targets are close to zero.
